# Speechify n8n Workflow Documentation

**System:** n8n (workflow automation platform)  
**Integration:** Brevo (formerly Sendinblue) email marketing API  
**Overall Purpose:** Automate outbound email campaigns targeting disability services staff at colleges and universities across 5 U.S. states (California, New York, Illinois, Florida, Texas). Each state receives a 5-email drip sequence with state-specific messaging.

---

## Campaign Structure Overview

All workflows operate against the same 25 Brevo email campaigns (IDs 192–216), organized as:

| Campaign ID | State | Email # | Angle |
|-------------|-------|---------|-------|
| 192 | California | E1 | Accessibility / DRC Pilot |
| 193 | California | E2 | Procurement |
| 194 | California | E3 | IT Security |
| 195 | California | E4 | Student Retention |
| 196 | California | E5 | Faculty Remediation |
| 197 | New York | E1 | Accessibility |
| 198 | New York | E2 | Procurement |
| 199 | New York | E3 | IT Security |
| 200 | New York | E4 | Student Retention |
| 201 | New York | E5 | Faculty Remediation |
| 202 | Illinois | E1 | Accessibility |
| 203 | Illinois | E2 | Procurement |
| 204 | Illinois | E3 | IT Security |
| 205 | Illinois | E4 | Student Retention |
| 206 | Illinois | E5 | Faculty Remediation |
| 207 | Florida | E1 | Accessibility |
| 208 | Florida | E2 | Procurement |
| 209 | Florida | E3 | IT Security |
| 210 | Florida | E4 | Student Retention |
| 211 | Florida | E5 | Faculty Remediation |
| 212 | Texas | E1 | Accessibility |
| 213 | Texas | E2 | Procurement |
| 214 | Texas | E3 | IT Security |
| 215 | Texas | E4 | Student Retention |
| 216 | Texas | E5 | Faculty Remediation |

**Brevo List IDs:** 12 = California, 13 = New York, 14 = Illinois, 15 = Florida, 16 = Texas

---

---
## Workflow 1: `n8n_speechify_campaign.json`
**n8n Name:** Speechify Campaign  
**System:** n8n → Brevo API  
**Purpose:** The **master campaign launcher**. Imports all contact lists into Brevo for all 5 states, then creates the 25 email campaigns with full HTML content, subjects, preview text, and scheduled send times.

### How It Works
- **Trigger:** Schedule Trigger — fires daily at **12:00 PM**
- **Step 1 — Import Contacts:** POSTs to `https://api.brevo.com/v3/contacts/import` in multiple batches per state. Each contact includes: email, FIRSTNAME, LASTNAME, SCHOOL, JOBTITLE, and the appropriate list ID.
- **Step 2 — Create Campaigns:** POSTs to `https://api.brevo.com/v3/emailCampaigns` to create each of the 25 campaigns with full HTML email content, sender info (Matt Basile, mattia@speechify.com), and a scheduled send time of `2026-06-12T12:00:00-05:00`.
- **Authentication:** HTTP Header Auth with Brevo API key

### Contact Batches
| Node | State | List ID | Approx. Contacts |
|------|-------|---------|------------------|
| Import CA Batch 1 | California | 12 | ~100 |
| Import CA Batch 2 | California | 12 | ~100 |
| Import CA Batch 3 | California | 12 | ~100 |
| Import NY Batch 1 | New York | 13 | ~100 |
| Import NY Batch 2 | New York | 13 | ~100 |
| Import IL Batch 1 | Illinois | 14 | ~100 |
| Import FL Batch 1 | Florida | 15 | ~100 |
| Import FL Batch 2 | Florida | 15 | ~100 |
| Import TX Batch 1 | Texas | 16 | ~50 |

### Flow Diagram
```
Schedule Trigger (12PM daily)
    ↓ (parallel branches per state)
Import CA Batch 1 → CA E1 Accessibility → CA E2 Procurement → CA E3 IT Security → CA E4 Retention → CA E5 Faculty
Import NY Batch 1 → NY E1 → NY E2 → NY E3 → NY E4 → NY E5
Import IL Batch 1 → IL E1 → IL E2 → IL E3 → IL E4 → IL E5
Import FL Batch 1 → FL E1 → FL E2 → FL E3 → FL E4 → FL E5
Import TX Batch 1 → TX E1 → TX E2 → TX E3 → TX E4 → TX E5
```

In [ ]:
# Workflow 1: Speechify Campaign — Core Logic Summary
# File: n8n_speechify_campaign.json
# System: n8n  |  Purpose: Import contacts + create all 25 email campaigns in Brevo

import json

BREVO_BASE_URL = "https://api.brevo.com/v3"
SENDER = {"name": "Matt Basile", "email": "mattia@speechify.com"}
SCHEDULE_TIME = "2026-06-12T12:00:00-05:00"

# State → Brevo List ID mapping
STATE_LIST_IDS = {
    "California": 12,
    "New York":   13,
    "Illinois":   14,
    "Florida":    15,
    "Texas":      16,
}

# The 5 email angles used for every state
EMAIL_ANGLES = [
    "E1 Accessibility",
    "E2 Procurement",
    "E3 IT Security",
    "E4 Retention",
    "E5 Faculty",
]

# Step 1: Import contacts via POST /contacts/import
# Payload structure per contact:
example_contact = {
    "email": "director@university.edu",
    "attributes": {
        "FIRSTNAME": "Jane",
        "LASTNAME": "Doe",
        "SCHOOL": "University Name",
        "JOBTITLE": "Director, Disability Resource Center"
    },
    "listIds": [12],  # 12 = California list
    "updateEnabled": True
}

# Step 2: Create campaign via POST /emailCampaigns
# Payload structure:
example_campaign_payload = {
    "name": "California E1 Accessibility",
    "subject": "DRC pilot — no IT queue, no paperwork",
    "previewText": "Caltech and UC San Bernardino are already in. 30-day pilot, no setup required.",
    "sender": SENDER,
    "type": "classic",
    "htmlContent": "<p>Hi {{params.FIRSTNAME}}, ...</p>",
    "recipients": {"listIds": [12]},
    "scheduledAt": SCHEDULE_TIME
}

print("Workflow 1 — Speechify Campaign")
print(f"States targeted: {list(STATE_LIST_IDS.keys())}")
print(f"Email angles per state: {EMAIL_ANGLES}")
print(f"Total campaigns to create: {len(STATE_LIST_IDS) * len(EMAIL_ANGLES)}")
print(f"Scheduled send time: {SCHEDULE_TIME}")
print(f"Sender: {SENDER['name']} <{SENDER['email']}>")

---
## Workflow 2: `n8n_speechify_final_http_georgia_commas_12pm.json`
**n8n Name:** Speechify FINAL HTTP Georgia Commas 12PM  
**System:** n8n → Brevo API  
**Purpose:** **Bulk update all 25 existing campaigns** with corrected HTML (Georgia serif font, proper comma usage instead of em-dashes, fixed spacing). Runs in sequence without throttling delays.

### How It Works
- **Trigger:** Manual Trigger (run on demand)
- **Method:** PUT to `https://api.brevo.com/v3/emailCampaigns/{id}` for each of the 25 campaigns
- **What it updates:** Subject line, preview text, HTML content (with Georgia font, corrected punctuation)
- **Execution:** Sequential — each update node connects directly to the next with no wait
- **Note:** This is the "Georgia Commas" version — uses commas instead of em-dashes in email copy for better email client compatibility

### Flow Diagram
```
Manual Trigger
    → Update CA E1 (ID 192) → Update CA E3 (ID 194) → Update CA E5 (ID 196)
    → Update NY E1 (ID 197) → Update NY E2 (ID 198) → ... → Update TX E5 (ID 216)
```
*(Note: Workflow 2 skips CA E2 and CA E4 — only updates the odd/key emails initially)*

In [ ]:
# Workflow 2: Speechify FINAL HTTP Georgia Commas 12PM
# File: n8n_speechify_final_http_georgia_commas_12pm.json
# System: n8n  |  Purpose: Bulk update campaigns (no throttle, Georgia font, commas)

# Campaign IDs updated in this workflow (sequential, no wait nodes)
campaign_updates = [
    {"id": 192, "name": "California E1 Accessibility"},
    {"id": 194, "name": "California E3 IT Security"},
    {"id": 196, "name": "California E5 Faculty"},
    {"id": 197, "name": "New York E1 Accessibility"},
    {"id": 198, "name": "New York E2 Procurement"},
    {"id": 199, "name": "New York E3 IT Security"},
    {"id": 200, "name": "New York E4 Retention"},
    {"id": 201, "name": "New York E5 Faculty"},
    {"id": 202, "name": "Illinois E1 Accessibility"},
    {"id": 203, "name": "Illinois E2 Procurement"},
    {"id": 204, "name": "Illinois E3 IT Security"},
    {"id": 205, "name": "Illinois E4 Retention"},
    {"id": 206, "name": "Illinois E5 Faculty"},
    {"id": 207, "name": "Florida E1 Accessibility"},
    {"id": 208, "name": "Florida E2 Procurement"},
    {"id": 209, "name": "Florida E3 IT Security"},
    {"id": 210, "name": "Florida E4 Retention"},
    {"id": 211, "name": "Florida E5 Faculty"},
    {"id": 212, "name": "Texas E1 Accessibility"},
    {"id": 213, "name": "Texas E2 Procurement"},
    {"id": 214, "name": "Texas E3 IT Security"},
    {"id": 215, "name": "Texas E4 Retention"},
    {"id": 216, "name": "Texas E5 Faculty"},
]

# Key difference from Workflow 1: Uses PUT (update) not POST (create)
# HTML uses Georgia serif font for better email rendering:
GEORGIA_STYLE = 'font-family:Georgia,serif;font-size:14px;color:#1a1a1a;max-width:580px;margin:0 auto;padding:24px 20px;'

# Commas replace em-dashes in copy (e.g. "Last one from me, a different angle" vs "Last one from me — a different angle")

print("Workflow 2 — FINAL HTTP Georgia Commas 12PM")
print(f"Campaigns updated: {len(campaign_updates)}")
print(f"Execution mode: Sequential (no wait between updates)")
print(f"Font style: Georgia serif")
print(f"Punctuation: Commas (no em-dashes for email client compatibility)")
print("\nCampaigns updated:")
for c in campaign_updates:
    print(f"  ID {c['id']}: {c['name']}")

---
## Workflow 3: `n8n_speechify_throttled_georgia_commas_12pm.json`
**n8n Name:** Speechify THROTTLED Georgia Commas 12PM  
**System:** n8n → Brevo API  
**Purpose:** Same bulk update as Workflow 2 (Georgia font, commas), but **adds 3-second Wait nodes between each API call** to avoid hitting Brevo rate limits.

### How It Works
- **Trigger:** Manual Trigger
- **Method:** PUT to `/emailCampaigns/{id}` for all 25 campaigns
- **Key difference from Workflow 2:** Inserts a `Wait (3 seconds)` node between every update
- **Pattern:** `Update → Wait 3s → Update → Wait 3s → ...` (25 updates × 2 nodes = 50 nodes total)
- **Why throttling matters:** Brevo's API has rate limits; firing 25 PUT requests instantly can cause 429 errors

### Flow Diagram
```
Manual Trigger
    → Update CA E1 → Wait 3s → Update CA E2 → Wait 3s → Update CA E3 → Wait 3s
    → ... (all 25 campaigns with 3s gaps)
    → Update TX E5 → Wait 3s
```

In [ ]:
# Workflow 3: Speechify THROTTLED Georgia Commas 12PM
# File: n8n_speechify_throttled_georgia_commas_12pm.json
# System: n8n  |  Purpose: Rate-limited bulk update (3s waits between API calls)

WAIT_SECONDS = 3  # Wait between each Brevo API call
TOTAL_CAMPAIGNS = 25

# All 25 campaigns updated (IDs 192–216), in order CA → NY → IL → FL → TX
# Pattern per campaign:
def workflow_3_pattern(campaign_id, campaign_name, wait_seconds=3):
    """
    For each campaign:
    1. PUT /emailCampaigns/{id}  — update subject, previewText, htmlContent, scheduledAt
    2. Wait {wait_seconds} seconds (n8n Wait node)
    """
    return {
        "step_1": f"PUT /emailCampaigns/{campaign_id}  ({campaign_name})",
        "step_2": f"Wait {wait_seconds}s"
    }

# Example: show the pattern for the first 3 campaigns
sample_campaigns = [
    (192, "California E1 Accessibility"),
    (193, "California E2 Procurement"),
    (194, "California E3 IT Security"),
]

total_time_seconds = TOTAL_CAMPAIGNS * WAIT_SECONDS

print("Workflow 3 — THROTTLED Georgia Commas 12PM")
print(f"Wait between calls: {WAIT_SECONDS} seconds")
print(f"Total campaigns: {TOTAL_CAMPAIGNS}")
print(f"Estimated total runtime: ~{total_time_seconds}s ({total_time_seconds//60}m {total_time_seconds%60}s)")
print(f"\nExecution pattern (first 3 of 25):")
for cid, cname in sample_campaigns:
    steps = workflow_3_pattern(cid, cname)
    print(f"  → {steps['step_1']}")
    print(f"  → {steps['step_2']}")
print(f"  → ... (continues for all 25 campaigns)")

---
## Workflow 4: `n8n_speechify_final_update_test_12pm_utc.json`
**n8n Name:** Speechify FINAL Update + Test + 12PM UTC  
**System:** n8n → Brevo API  
**Purpose:** Updates all 25 campaigns AND **sends a test email to dylan@speechify.com** after each update, with 4-second throttling between pairs. Scheduled at **12PM UTC** (`2026-06-12T17:00:00.000Z`).

### How It Works
- **Trigger:** Manual Trigger
- **Per campaign:** PUT update → POST sendTest → Wait 4s → next campaign
- **Test recipient:** `dylan@speechify.com`
- **Schedule time:** `2026-06-12T17:00:00.000Z` (12PM UTC = 7AM ET / earlier than other workflows)
- **Why this version:** Allows team to QA each email before it goes live — Dylan receives all 25 test sends

### Flow Diagram
```
Manual Trigger
    → Update CA E1 → Send Test (→ dylan@speechify.com) → Wait 4s
    → Update CA E2 → Send Test → Wait 4s
    → ... (all 25 campaigns)
    → Update TX E5 → Send Test → Wait 4s
```

In [ ]:
# Workflow 4: Speechify FINAL Update + Test + 12PM UTC
# File: n8n_speechify_final_update_test_12pm_utc.json
# System: n8n  |  Purpose: Update + send test email per campaign (4s throttle)

WAIT_SECONDS = 4           # Slightly longer wait than Workflow 3
TEST_RECIPIENT = "dylan@speechify.com"
SCHEDULE_UTC = "2026-06-12T17:00:00.000Z"   # 12PM UTC = 7AM ET
TOTAL_CAMPAIGNS = 25

def workflow_4_pattern(campaign_id, campaign_name):
    """
    For each campaign:
    1. PUT /emailCampaigns/{id}   — update content + set scheduledAt to 12PM UTC
    2. POST /emailCampaigns/{id}/sendTest  — fire test email to dylan@speechify.com
    3. Wait 4 seconds
    """
    return [
        f"PUT  /emailCampaigns/{campaign_id}  (update: {campaign_name})",
        f"POST /emailCampaigns/{campaign_id}/sendTest  → {TEST_RECIPIENT}",
        f"Wait {WAIT_SECONDS}s"
    ]

# sendTest payload
send_test_payload = {"emailTo": [TEST_RECIPIENT]}

print("Workflow 4 — FINAL Update + Test + 12PM UTC")
print(f"Test emails sent to: {TEST_RECIPIENT}")
print(f"Scheduled send time: {SCHEDULE_UTC} (12PM UTC / 7AM ET)")
print(f"Wait between batches: {WAIT_SECONDS}s")
print(f"Total campaigns: {TOTAL_CAMPAIGNS}")
print(f"Total test emails sent: {TOTAL_CAMPAIGNS}")
print(f"\nPattern for California E1 (ID 192):")
for step in workflow_4_pattern(192, "California E1 Accessibility"):
    print(f"  → {step}")

---
## Workflow 5: `n8n_speechify_final_fixed_spacing_dylan_test.json`
**n8n Name:** Speechify FINAL Fixed Spacing + Dylan + Test  
**System:** n8n → Brevo API  
**Purpose:** Final production version. **Adds Dylan to all contact lists first**, then updates all 25 campaigns with fixed HTML spacing (`padding:20px` instead of `padding:24px 20px`), and sends test emails. Scheduled at **11AM ET** (`2026-06-12T16:00:00.000Z`).

### How It Works
- **Trigger:** Manual Trigger
- **Step 1 — Add Dylan to lists:** POST to `/contacts` to add `dylan@speechify.com` to all 5 state lists (12–16) sequentially
- **Step 2 — Update + Test:** Same pattern as Workflow 4 (update → sendTest → wait 4s) for all 25 campaigns
- **Schedule time:** `2026-06-12T16:00:00.000Z` (11AM ET)
- **HTML fix:** Padding changed from `24px 20px` to `20px` (uniform spacing)

### Flow Diagram
```
Manual Trigger
    → Add Dylan to CA (list 12) → Add Dylan to NY (list 13)
    → Add Dylan to IL (list 14) → Add Dylan to FL (list 15)
    → Add Dylan to TX (list 16)
    → Update CA E1 → Test CA E1 → Wait 4s
    → Update CA E2 → Test CA E2 → Wait 4s
    → ... (all 25 campaigns)
```

In [ ]:
# Workflow 5: Speechify FINAL Fixed Spacing + Dylan + Test
# File: n8n_speechify_final_fixed_spacing_dylan_test.json
# System: n8n  |  Purpose: Add Dylan to lists, update campaigns (fixed spacing), send tests

DYLAN_EMAIL = "dylan@speechify.com"
SCHEDULE_11AM_ET = "2026-06-12T16:00:00.000Z"   # 11AM ET
WAIT_SECONDS = 4

# Phase 1: Add Dylan as a test contact to all 5 state lists
dylan_contact = {
    "email": DYLAN_EMAIL,
    "attributes": {
        "FIRSTNAME": "Dylan",
        "LASTNAME": "Kur",
        "SCHOOL": "Speechify",
        "JOBTITLE": "Intern"
    },
    "updateEnabled": True
}

state_lists = [
    {"state": "California", "listId": 12},
    {"state": "New York",   "listId": 13},
    {"state": "Illinois",  "listId": 14},
    {"state": "Florida",   "listId": 15},
    {"state": "Texas",     "listId": 16},
]

# Phase 2: HTML spacing fix
OLD_PADDING = "padding:24px 20px"   # used in Workflows 2, 3, 4
NEW_PADDING = "padding:20px"         # fixed in Workflow 5

print("Workflow 5 — FINAL Fixed Spacing + Dylan + Test")
print(f"\nPhase 1: Add Dylan ({DYLAN_EMAIL}) to {len(state_lists)} state lists")
for sl in state_lists:
    print(f"  POST /contacts  listId={sl['listId']} ({sl['state']})")

print(f"\nPhase 2: Update all 25 campaigns + send test emails")
print(f"  HTML fix: '{OLD_PADDING}' → '{NEW_PADDING}'")
print(f"  Schedule: {SCHEDULE_11AM_ET} (11AM ET)")
print(f"  Test recipient: {DYLAN_EMAIL}")
print(f"  Throttle: {WAIT_SECONDS}s between each campaign")

---
## Workflow 6: `n8n_add_matt_to_all_lists.json`
**n8n Name:** Add Matt to All Lists  
**System:** n8n → Brevo API  
**Purpose:** A simple utility workflow — adds **Matt Basile (mattia@speechify.com)** as a contact to all 5 state lists in Brevo so he receives the actual campaign emails alongside real university contacts.

### How It Works
- **Trigger:** Manual Trigger
- **5 sequential nodes:** POST to `/contacts` for each state list
- **Use case:** Quality assurance — Matt can verify what real recipients see when campaigns go live

### Flow Diagram
```
Manual Trigger
    → Add Matt to CA (list 12)
    → Add Matt to NY (list 13)
    → Add Matt to IL (list 14)
    → Add Matt to FL (list 15)
    → Add Matt to TX (list 16)
```

In [ ]:
# Workflow 6: Add Matt to All Lists
# File: n8n_add_matt_to_all_lists.json
# System: n8n  |  Purpose: Add Matt Basile as a contact to all 5 state lists

MATT_EMAIL = "mattia@speechify.com"

matt_contact_template = {
    "email": MATT_EMAIL,
    "attributes": {
        "FIRSTNAME": "Matt",
        "LASTNAME": "Basile",
        "SCHOOL": "Speechify",
        "JOBTITLE": "Head of Education"
    },
    "updateEnabled": True
}

state_lists = [
    {"state": "California", "listId": 12, "node": "Add Matt to California"},
    {"state": "New York",   "listId": 13, "node": "Add Matt to New York"},
    {"state": "Illinois",  "listId": 14, "node": "Add Matt to Illinois"},
    {"state": "Florida",   "listId": 15, "node": "Add Matt to Florida"},
    {"state": "Texas",     "listId": 16, "node": "Add Matt to Texas"},
]

print("Workflow 6 — Add Matt to All Lists")
print(f"Contact: {matt_contact_template['attributes']['FIRSTNAME']} {matt_contact_template['attributes']['LASTNAME']}")
print(f"Email: {MATT_EMAIL}")
print(f"Role: {matt_contact_template['attributes']['JOBTITLE']}")
print(f"\nAdding to {len(state_lists)} lists:")
for sl in state_lists:
    payload = {**matt_contact_template, "listIds": [sl['listId']]}
    print(f"  {sl['node']}: POST /contacts  listId={sl['listId']} ({sl['state']})")

---
## Workflow 7: `n8n_speechify_proofread_test_schedule_all_25.json`
**n8n Name:** Speechify - Proofread + Test + Schedule All 25  
**System:** n8n → Brevo API  
**Purpose:** The most sophisticated workflow. Automatically **fetches all campaigns, proofreads them with JavaScript logic, builds a report, sends test emails, and schedules campaigns** — all in one run.

### How It Works
1. **Get All Campaigns:** GET `/emailCampaigns?limit=100` — retrieves all campaigns from Brevo
2. **Proofread & Extract (Code Node):** JavaScript filters for state campaigns and validates:
   - Subject line contains expected keyword fragment
   - Sender email is `mattia@speechify.com`
   - Recipients list is set
   - Preview text is not empty
   - Returns a `proofreadPassed: true/false` flag and list of `issues` per campaign
3. **Build Report (Code Node):** Iterates items, assigns a staggered scheduled time (every 5 days from now), and generates a report label per campaign
4. **Send Test Email:** POST `/emailCampaigns/{id}/sendTest` → `dylan@speechify.com`
5. **Schedule Campaign:** PUT `/emailCampaigns/{id}` with the computed `scheduledAt` time

### Flow Diagram
```
Manual Trigger
    → GET /emailCampaigns (fetch all)
    → Proofread & Extract (JS: filter state campaigns, validate subject/sender/recipients/preview)
    → Build Report (JS: assign staggered schedules, generate report)
    ↓ (parallel outputs)
    → Send Test Email (POST sendTest → dylan@speechify.com)
    → Schedule Campaign (PUT with scheduledAt)
```

### Proofread Validation Rules
| Check | Expected Value |
|-------|---------------|
| Subject fragment | e.g. `"DRC pilot"` for CA E1, `"Vendor packet"` for E2 |
| Sender email | `mattia@speechify.com` |
| Recipients | At least one list assigned |
| Preview text | Non-empty string |

In [ ]:
# Workflow 7: Speechify - Proofread + Test + Schedule All 25
# File: n8n_speechify_proofread_test_schedule_all_25.json
# System: n8n  |  Purpose: Automated QA + test send + scheduling for all 25 campaigns

from datetime import datetime, timedelta

# ── Proofread & Extract Logic (mirrors the n8n Code node JS) ────────────────

STATE_NAMES = ['California', 'New York', 'Illinois', 'Florida', 'Texas']

# Expected subject keyword fragments per campaign name
EXPECTED_SUBJECTS = {
    'California E1 Accessibility': 'DRC pilot',
    'California E2 Procurement':   'Vendor packet',
    'California E3 IT Security':   'Shadow IT',
    'California E4 Retention':     'Reading wall hitting CA',
    'California E5 Faculty':       'Where CA remediation',
    'New York E1 Accessibility':   'NY disability offices',
    'New York E2 Procurement':     'Vendor packet',
    'New York E3 IT Security':     'Shadow IT',
    'New York E4 Retention':       'Reading wall hitting NY',
    'New York E5 Faculty':         'Where NY remediation',
    'Illinois E1 Accessibility':   'Illinois DRC pilot',
    'Illinois E2 Procurement':     'Vendor packet',
    'Illinois E3 IT Security':     'Browser extensions',
    'Illinois E4 Retention':       'Where IL freshmen',
    'Illinois E5 Faculty':         'Where IL remediation',
    'Florida E1 Accessibility':    'Florida DRC pilot',
    'Florida E2 Procurement':      'Vendor packet',
    'Florida E3 IT Security':      'Shadow IT',
    'Florida E4 Retention':        'Reading wall hitting FL',
    'Florida E5 Faculty':          'Where FL remediation',
    'Texas E1 Accessibility':      'Texas DRC pilot',
    'Texas E2 Procurement':        'Vendor packet',
    'Texas E3 IT Security':        'TTS with an audit trail',
    'Texas E4 Retention':          'Reading wall hitting TX',
    'Texas E5 Faculty':            'Where TX remediation',
}

def proofread_campaign(campaign: dict) -> dict:
    """Validate a Brevo campaign object against expected values."""
    name = campaign.get('name', '')
    expected_fragment = EXPECTED_SUBJECTS.get(name, '')
    
    issues = []
    subject = campaign.get('subject', '')
    sender = campaign.get('sender', {})
    recipients = campaign.get('recipients', {})
    preview = campaign.get('previewText', '')
    
    if expected_fragment and expected_fragment not in subject:
        issues.append(f'Subject mismatch: got "{subject}", expected fragment "{expected_fragment}"')
    if sender.get('email') != 'mattia@speechify.com':
        issues.append(f'Wrong sender: {sender}')
    if not recipients.get('lists'):
        issues.append('No recipients list assigned')
    if not preview:
        issues.append('Missing preview text / preheader')
    
    return {
        'name': name,
        'proofreadPassed': len(issues) == 0,
        'issues': issues
    }

def build_schedule(campaigns: list, base_date: datetime = None) -> list:
    """Assign staggered schedule times (every 5 days) to a list of campaigns."""
    if base_date is None:
        base_date = datetime.utcnow()
    scheduled = []
    for i, c in enumerate(campaigns):
        send_time = base_date + timedelta(days=i * 5)
        send_time = send_time.replace(hour=12, minute=0, second=0, microsecond=0)
        scheduled.append({**c, 'scheduledAt': send_time.isoformat()})
    return scheduled

# ── Demo run ─────────────────────────────────────────────────────────────────

# Simulate what Brevo returns for campaign 192
mock_campaign = {
    'id': 192,
    'name': 'California E1 Accessibility',
    'subject': 'DRC pilot, no IT queue, no paperwork',
    'previewText': 'Caltech and UC San Bernardino are already in.',
    'sender': {'name': 'Matt Basile', 'email': 'mattia@speechify.com'},
    'recipients': {'lists': [{'id': 12}]},
    'status': 'draft'
}

result = proofread_campaign(mock_campaign)
print("Workflow 7 — Proofread + Test + Schedule All 25")
print(f"\nProofread result for '{result['name']}':")
print(f"  Passed: {result['proofreadPassed']}")
print(f"  Issues: {result['issues'] if result['issues'] else 'None'}")

# Show staggered schedule for first 5 campaigns
mock_campaigns = [{"name": name, "id": 192+i} for i, name in enumerate(list(EXPECTED_SUBJECTS.keys())[:5])]
scheduled = build_schedule(mock_campaigns, base_date=datetime(2026, 6, 12, 12, 0, 0))
print(f"\nStaggered schedule (first 5 campaigns, 5-day gaps):")
for c in scheduled:
    print(f"  {c['name']}: {c['scheduledAt']}")

---
## Summary: Workflow Evolution & Relationships

These 7 workflows represent an **iterative build-test-refine cycle** for a Speechify B2B email campaign targeting higher education disability services offices.

| # | Workflow | Role | Trigger | Key Feature |
|---|----------|------|---------|-------------|
| 1 | Speechify Campaign | **Create** contacts + campaigns | Schedule (12PM daily) | Imports all contacts; creates all 25 campaigns |
| 2 | FINAL HTTP Georgia Commas 12PM | **Update** campaigns | Manual | No throttle; Georgia font; comma punctuation |
| 3 | THROTTLED Georgia Commas 12PM | **Update** (rate-limited) | Manual | 3s waits to avoid API rate limits |
| 4 | FINAL Update + Test + 12PM UTC | **Update + QA** | Manual | Adds test emails to Dylan; 4s waits; 12PM UTC |
| 5 | FINAL Fixed Spacing + Dylan + Test | **Update + QA + Setup** | Manual | Adds Dylan to lists first; fixes HTML padding; 11AM ET |
| 6 | Add Matt to All Lists | **Setup** | Manual | Adds Matt as contact to all 5 state lists |
| 7 | Proofread + Test + Schedule All 25 | **QA + Schedule** | Manual | Auto-fetches, validates, tests, and schedules all |

### Recommended Execution Order for a New Campaign
```
1. Run Workflow 6 (Add Matt to All Lists)         → so Matt gets the real emails
2. Run Workflow 1 (Speechify Campaign)             → imports contacts + creates campaigns
3. Run Workflow 5 (Fixed Spacing + Dylan + Test)   → QA test sends + final HTML fixes
4. Run Workflow 7 (Proofread + Test + Schedule)    → automated validation + scheduling
```

### API Endpoints Used
| Endpoint | Method | Used For |
|----------|--------|----------|
| `/v3/contacts` | POST | Add single contact to a list |
| `/v3/contacts/import` | POST | Bulk import contacts |
| `/v3/emailCampaigns` | POST | Create a new campaign |
| `/v3/emailCampaigns` | GET | Fetch all campaigns |
| `/v3/emailCampaigns/{id}` | PUT | Update existing campaign |
| `/v3/emailCampaigns/{id}/sendTest` | POST | Send test email |

In [ ]:
# Summary: Print a quick reference of all 7 workflows

workflows = [
    {
        "num": 1,
        "file": "n8n_speechify_campaign.json",
        "name": "Speechify Campaign",
        "trigger": "Schedule (12PM daily)",
        "purpose": "Import contacts + create all 25 campaigns",
        "api_calls": ["POST /contacts/import", "POST /emailCampaigns"]
    },
    {
        "num": 2,
        "file": "n8n_speechify_final_http_georgia_commas_12pm.json",
        "name": "FINAL HTTP Georgia Commas 12PM",
        "trigger": "Manual",
        "purpose": "Bulk update campaigns (no throttle, Georgia font)",
        "api_calls": ["PUT /emailCampaigns/{id}"]
    },
    {
        "num": 3,
        "file": "n8n_speechify_throttled_georgia_commas_12pm.json",
        "name": "THROTTLED Georgia Commas 12PM",
        "trigger": "Manual",
        "purpose": "Bulk update campaigns with 3s throttle between calls",
        "api_calls": ["PUT /emailCampaigns/{id}", "Wait 3s"]
    },
    {
        "num": 4,
        "file": "n8n_speechify_final_update_test_12pm_utc.json",
        "name": "FINAL Update + Test + 12PM UTC",
        "trigger": "Manual",
        "purpose": "Update + send test emails (dylan@speechify.com), 4s throttle",
        "api_calls": ["PUT /emailCampaigns/{id}", "POST /emailCampaigns/{id}/sendTest", "Wait 4s"]
    },
    {
        "num": 5,
        "file": "n8n_speechify_final_fixed_spacing_dylan_test.json",
        "name": "FINAL Fixed Spacing + Dylan + Test",
        "trigger": "Manual",
        "purpose": "Add Dylan to lists, fix HTML spacing, update + test all 25",
        "api_calls": ["POST /contacts", "PUT /emailCampaigns/{id}", "POST /emailCampaigns/{id}/sendTest"]
    },
    {
        "num": 6,
        "file": "n8n_add_matt_to_all_lists.json",
        "name": "Add Matt to All Lists",
        "trigger": "Manual",
        "purpose": "Add Matt Basile as contact to all 5 state lists",
        "api_calls": ["POST /contacts (×5)"]
    },
    {
        "num": 7,
        "file": "n8n_speechify_proofread_test_schedule_all_25.json",
        "name": "Proofread + Test + Schedule All 25",
        "trigger": "Manual",
        "purpose": "Auto-fetch, validate, test, and schedule all 25 campaigns",
        "api_calls": ["GET /emailCampaigns", "POST /sendTest", "PUT /emailCampaigns/{id}"]
    },
]

print("=" * 70)
print("SPEECHIFY n8n WORKFLOW REFERENCE")
print("=" * 70)
for wf in workflows:
    print(f"\nWorkflow {wf['num']}: {wf['name']}")
    print(f"  File:    {wf['file']}")
    print(f"  Trigger: {wf['trigger']}")
    print(f"  Purpose: {wf['purpose']}")
    print(f"  API:     {', '.join(wf['api_calls'])}")
print("\n" + "=" * 70)